In [ ]:
import os
import time
import psycopg2
import duckdb

# Ajustar esta ruta si es necesario 
DATA_DIR = os.getcwd()
DATA_DIR

In [2]:
nombre_bd = #poner nombre aca
conn_pg = psycopg2.connect(dbname="nombre_bd")
conn_pg.autocommit = True
cur = conn_pg.cursor()

# Crear tablas
cur.execute("""
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
""")

cur.execute("""
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    brand        TEXT,
    base_price   NUMERIC(10,2),
    weight_kg    NUMERIC(10,2),
    rating       NUMERIC(3,1)
);

CREATE TABLE customers (
    customer_id       INTEGER PRIMARY KEY,
    customer_name     TEXT,
    region            TEXT,
    city              TEXT,
    segment           TEXT,
    registration_date DATE
);

CREATE TABLE sales (
    sale_id        INTEGER PRIMARY KEY,
    sale_date      DATE,
    sale_time      TIME,
    product_id     INTEGER,
    customer_id    INTEGER,
    quantity       INTEGER,
    unit_price     NUMERIC(10,2),
    discount_pct   NUMERIC(5,2),
    tax_pct        NUMERIC(5,2),
    amount         NUMERIC(12,2),
    payment_method TEXT,
    channel        TEXT,
    status         TEXT,
    shipping_cost  NUMERIC(10,2),
    shipping_days  INTEGER,
    warehouse_id   INTEGER,
    salesperson_id INTEGER,
    promotion_id   INTEGER,
    notes          TEXT,
    is_gift        BOOLEAN
);
""")

# Cargar CSVs (COPY es mucho más rápido que INSERT)
for table in ["products", "customers", "sales"]:
    csv_path = os.path.join(DATA_DIR, f"{table}.csv")
    with open(csv_path, "r") as f:
        cur.copy_expert(f"COPY {table} FROM STDIN WITH CSV HEADER NULL ''", f)

# Actualizar estadísticas del optimizador
cur.execute("ANALYZE;")
print("PostgreSQL: datos cargados")

PostgreSQL: datos cargados


In [3]:
conn_duck = duckdb.connect(nombre_bd+".duckdb")

for table in ["products", "customers", "sales"]:
    csv_path = os.path.join(DATA_DIR, f"{table}.csv")
    conn_duck.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM read_csv('{csv_path}')")

# DuckDB calcula estadísticas automáticamente — no necesita ANALYZE manual
print("DuckDB: datos cargados")

DuckDB: datos cargados


In [4]:
def explain_pg(query, analyze=True):
    """Ejecuta EXPLAIN (ANALYZE, BUFFERS) en PostgreSQL y muestra el resultado."""
    prefix = "EXPLAIN (ANALYZE, BUFFERS)" if analyze else "EXPLAIN"
    cur.execute(f"{prefix} {query}")
    for row in cur.fetchall():
        print(row[0])
    print()

def explain_duck(query, analyze=True):
    """Ejecuta EXPLAIN (ANALYZE) en DuckDB y muestra el resultado."""
    prefix = "EXPLAIN ANALYZE" if analyze else "EXPLAIN"
    result = conn_duck.execute(f"{prefix} {query}").fetchall()
    for row in result:
        print(row[-1] if len(row) > 1 else row[0])
    print()

In [5]:
# Consulta poco selectiva — ¿qué scan elige?
explain_pg("SELECT * FROM sales WHERE amount > 100;")


Seq Scan on sales  (cost=0.00..30060.00 rows=833697 width=103) (actual time=0.280..121.683 rows=832227.00 loops=1)
  Filter: (amount > '100'::numeric)
  Rows Removed by Filter: 167773
  Buffers: shared hit=2558 read=15002
Planning:
  Buffers: shared hit=103
Planning Time: 2.374 ms
Execution Time: 146.842 ms



In [6]:
# Consulta MUY selectiva — ¿qué scan elige?
explain_pg("SELECT * FROM sales WHERE sale_id = 42;")


Index Scan using sales_pkey on sales  (cost=0.42..8.44 rows=1 width=103) (actual time=0.202..0.203 rows=1.00 loops=1)
  Index Cond: (sale_id = 42)
  Index Searches: 1
  Buffers: shared hit=3 read=1
Planning:
  Buffers: shared hit=5
Planning Time: 0.960 ms
Execution Time: 0.582 ms

